<a href="https://colab.research.google.com/github/GulshanKumar0/Gulshan07/blob/main/LanguageTranslation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [38]:
# Sample English-Hindi sentence pairs
english_sentences = [
    "hello", "how are you", "what is your name", "good morning",
    "good night", "thank you", "please", "sorry", "yes", "no",
    "where are you", "I am fine", "what do you do", "I love you",
    "goodbye", "see you later", "take care", "come here", "go there", "help me"
]

hindi_sentences = [
    "नमस्ते", "आप कैसे हैं", "आपका नाम क्या है", "सुप्रभात",
    "शुभ रात्रि", "धन्यवाद", "कृपया", "मुझे माफ़ करें", "हाँ", "नहीं",
    "आप कहाँ हैं", "मैं ठीक हूँ", "आप क्या करते हैं", "मैं तुमसे प्यार करता हूँ",
    "अलविदा", "फिर मिलेंगे", "ध्यान रखना", "यहाँ आओ", "वहाँ जाओ", "मेरी मदद करो"
]

# Tokenize the sentences
def tokenize(sentences):
    tokenizer = Tokenizer()
    tokenizer.fit_on_texts(sentences)
    sequences = tokenizer.texts_to_sequences(sentences)
    return tokenizer, sequences

# Tokenizing English sentences
eng_tokenizer, eng_sequences = tokenize(english_sentences)
eng_vocab_size = len(eng_tokenizer.word_index) + 1
max_eng_len = max([len(seq) for seq in eng_sequences])

# Tokenizing Hindi sentences
hin_tokenizer, hin_sequences = tokenize(hindi_sentences)
hin_vocab_size = len(hin_tokenizer.word_index) + 1
max_hin_len = max([len(seq) for seq in hin_sequences])

# Padding sequences
eng_sequences = pad_sequences(eng_sequences, maxlen=max_eng_len, padding='post')
hin_sequences = pad_sequences(hin_sequences, maxlen=max_hin_len, padding='post')

# Prepare decoder input and target data
decoder_input_data = hin_sequences[:, :-1]  # Remove the last token for input
decoder_target_data = hin_sequences[:, 1:]  # Remove the first token for target

# Expand dimensions of target data to match the model output shape
decoder_target_data = np.expand_dims(decoder_target_data, -1)


In [39]:
# Encoder
encoder_inputs = Input(shape=(max_eng_len,))
encoder_embedding = Embedding(eng_vocab_size, 256, mask_zero=True)(encoder_inputs)
encoder_lstm = LSTM(256, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(max_hin_len-1,))
decoder_embedding = Embedding(hin_vocab_size, 256, mask_zero=True)(decoder_inputs)
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)
decoder_dense = Dense(hin_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Define the model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8             │ (None, 4)              │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ input_layer_9             │ (None, 4)              │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_4 (Embedding)   │ (None, 4, 256)         │          8,704 │ input_layer_8[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ not_equal_4 (NotEqual)    │ (None, 4)              │              0 │ input_layer_8[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ embedding_5 (Embedding)   │ (None, 4, 256)         │          9,984 │ input_layer_9[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_4 (LSTM)             │ [(None, 256), (None,   │        525,312 │ embedding_4[0][0],     │
│                           │ 256), (None, 256)]     │                │ not_equal_4[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_5 (LSTM)             │ [(None, 4, 256),       │        525,312 │ embedding_5[0][0],     │
│                           │ (None, 256), (None,    │                │ lstm_4[0][1],          │
│                           │ 256)]                  │                │ lstm_4[0][2]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_2 (Dense)           │ (None, 4, 39)          │         10,023 │ lstm_5[0][0]           │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 1,079,335 (4.12 MB)

 Trainable params: 1,079,335 (4.12 MB)

 Non-trainable params: 0 (0.00 B)

In [40]:
model.fit([eng_sequences, decoder_input_data], decoder_target_data,
          batch_size=64, epochs=200, validation_split=0.2)


Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.0000e+00 - loss: 3.6635 - val_accuracy: 0.1250 - val_loss: 3.6632
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - accuracy: 0.4375 - loss: 3.6365 - val_accuracy: 0.5625 - val_loss: 3.6578
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.7188 - loss: 3.6083 - val_accuracy: 0.5625 - val_loss: 3.6520
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step - accuracy: 0.7031 - loss: 3.5769 - val_accuracy: 0.5625 - val_loss: 3.6455
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.7031 - loss: 3.5407 - val_accuracy: 0.6875 - val_loss: 3.6381
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.6875 - loss: 3.4974 - val_accuracy: 0.6875 - val_loss: 3.6295
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.6875 - loss: 3.4445 - val_accuracy: 0.6875 - val_loss: 3.6195
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - accuracy: 0.6875 - loss: 3.3785 - val_accuracy: 0.6875 - v

In [41]:
# Encoder model for inference
encoder_model = Model(encoder_inputs, encoder_states)

# Decoder model for inference
decoder_state_input_h = Input(shape=(256,))
decoder_state_input_c = Input(shape=(256,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_embedding, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states)

# Translation function


In [42]:
def translate_sentence(input_seq):
    states_value = encoder_model.predict(input_seq)

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = hin_tokenizer.word_index['नमस्ते']  # Assuming 'नमस्ते' as the start token

    stop_condition = False
    decoded_sentence = ''
    previous_words = []

    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = None

        # Find the word corresponding to the predicted index
        for word, index in hin_tokenizer.word_index.items():
            if index == sampled_token_index:
                sampled_word = word
                break

        # If the word is None or if it's already in previous words, stop the loop
        if sampled_word is None or (previous_words and sampled_word == previous_words[-1]):
            break

        decoded_sentence += ' ' + sampled_word
        previous_words.append(sampled_word)

        # Exit condition: stop if end token is reached or length exceeds max length
        if sampled_word == 'अलविदा' or len(decoded_sentence.split()) > max_hin_len:
            stop_condition = True

        # Update the target sequence with the predicted word
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index

        # Update states
        states_value = [h, c]

    return decoded_sentence.strip()


In [43]:
input_sentence = "how are you"
input_sequence = eng_tokenizer.texts_to_sequences([input_sentence])
input_sequence = pad_sequences(input_sequence, maxlen=max_eng_len, padding='post')

translated_sentence = translate_sentence(input_sequence)
print(f"Input: {input_sentence}")
print(f"Translated: {translated_sentence}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 211ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Input: how are you
Translated: कैसे


In [26]:
from random import choice

# Original English-Hindi sentence pairs
english_sentences = [
    "hello", "how are you", "what is your name", "good morning",
    "good night", "thank you", "please", "sorry", "yes", "no",
    "where are you", "I am fine", "what do you do", "I love you",
    "goodbye", "see you later", "take care", "come here", "go there", "help me"
]

hindi_sentences = [
    "नमस्ते", "आप कैसे हैं", "आपका नाम क्या है", "सुप्रभात",
    "शुभ रात्रि", "धन्यवाद", "कृपया", "मुझे माफ़ करें", "हाँ", "नहीं",
    "आप कहाँ हैं", "मैं ठीक हूँ", "आप क्या करते हैं", "मैं तुमसे प्यार करता हूँ",
    "अलविदा", "फिर मिलेंगे", "ध्यान रखना", "यहाँ आओ", "वहाँ जाओ", "मेरी मदद करो"
]

# Function to generate variations of sentences
def generate_variations(sentences, multiplier=5):
    variations = []
    for sentence in sentences:
        for _ in range(multiplier):
            variations.append(sentence + " " + choice(["please", "thank you", "sorry"]))
    return variations

# Expand the dataset
num_sentences = 100
current_size = len(english_sentences)

# Create additional samples
while current_size < num_sentences:
    new_sentences = generate_variations(english_sentences, multiplier=2)
    english_sentences.extend(new_sentences)
    hindi_sentences.extend(generate_variations(hindi_sentences, multiplier=2))
    current_size = len(english_sentences)

# Trim to exact size
english_sentences = english_sentences[:num_sentences]
hindi_sentences = hindi_sentences[:num_sentences]

# Check if the length matches
assert len(english_sentences) == len(hindi_sentences) == num_sentences

print(f"Number of English sentences: {len(english_sentences)}")
print(f"Number of Hindi sentences: {len(hindi_sentences)}")


Number of English sentences: 100
Number of Hindi sentences: 100


In [21]:
english_sentences

['hello',
 'how are you',
 'what is your name',
 'good morning',
 'good night',
 'thank you',
 'please',
 'sorry',
 'yes',
 'no',
 'where are you',
 'I am fine',
 'what do you do',
 'I love you',
 'goodbye',
 'see you later',
 'take care',
 'come here',
 'go there',
 'help me',
 'hello thank you',
 'hello thank you',
 'how are you thank you',
 'how are you thank you',
 'what is your name thank you',
 'what is your name please',
 'good morning please',
 'good morning thank you',
 'good night thank you',
 'good night thank you',
 'thank you sorry',
 'thank you please',
 'please sorry',
 'please thank you',
 'sorry thank you',
 'sorry sorry',
 'yes thank you',
 'yes sorry',
 'no please',
 'no sorry',
 'where are you sorry',
 'where are you please',
 'I am fine please',
 'I am fine thank you',
 'what do you do please',
 'what do you do thank you',
 'I love you thank you',
 'I love you sorry',
 'goodbye thank you',
 'goodbye please',
 'see you later please',
 'see you later please',
 'take 

In [32]:
input_sentence = "My Name is Gulshan Kumar"
input_sequence = eng_tokenizer.texts_to_sequences([input_sentence])
input_sequence = pad_sequences(input_sequence, maxlen=max_eng_len, padding='post')

translated_sentence = translate_sentence(input_sequence)
print(f"Input: {input_sentence}")
print(f"Translated: {translated_sentence}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Input: My Name is Gulshan Kumar
Translated: 


In [33]:
input_sentence = "Hello "
input_sequence = eng_tokenizer.texts_to_sequences([input_sentence])
input_sequence = pad_sequences(input_sequence, maxlen=max_eng_len, padding='post')

translated_sentence = translate_sentence(input_sequence)
print(f"Input: {input_sentence}")
print(f"Translated: {translated_sentence}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
Input: Hello 
Translated: 


In [34]:
input_sentence = "what is your name "
input_sequence = eng_tokenizer.texts_to_sequences([input_sentence])
input_sequence = pad_sequences(input_sequence, maxlen=max_eng_len, padding='post')

translated_sentence = translate_sentence(input_sequence)
print(f"Input: {input_sentence}")
print(f"Translated: {translated_sentence}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Input: what is your name 
Translated: नाम क्या है
